# Exercises XP: Diabetes Prediction with Logistic Regression

**Course:** Developers Institute  **Week 5 - Day 1**  
**Author:** Alex Goldbaum

Goal: build a Logistic Regression model that predicts whether an individual has
diabetes, evaluate it with standard classification metrics, visualize its decision
boundary in 2D, and plot the ROC curve.

Dataset: **Pima Indians Diabetes** (768 rows, 8 features, binary `Outcome` target).


## Setup — imports + load the dataset


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_score, recall_score, f1_score,
    roc_curve, roc_auc_score, ConfusionMatrixDisplay,
)
from sklearn.decomposition import PCA

sns.set_theme(style='whitegrid')

URL = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
COLUMNS = [
    'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
    'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome',
]

df = pd.read_csv(URL, header=None, names=COLUMNS)
print('Shape:', df.shape)
df.head()


## Exercise 1: Understanding the problem and Data Collection

We want to **predict whether an individual has diabetes** (`Outcome = 1`) given
8 clinical features (Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin,
BMI, DiabetesPedigreeFunction, Age). This is a **binary classification** problem.


In [ ]:
# Basic exploration
print('Dataset shape:', df.shape)
print('\nFeature dtypes:')
print(df.dtypes)
print('\nStatistical summary:')
df.describe()


In [ ]:
# How many positive (diabetic) vs negative (non-diabetic) cases?
counts = df['Outcome'].value_counts().rename({0: 'Negative (no diabetes)', 1: 'Positive (diabetes)'})
print('Class distribution:')
print(counts)
print(f'\nClass balance: {counts.iloc[1] / counts.sum() * 100:.1f}% positive')

# Visualize
plt.figure(figsize=(6, 4))
sns.countplot(x='Outcome', data=df, palette=['steelblue', 'tomato'])
plt.xticks([0, 1], ['Negative', 'Positive'])
plt.title('Class distribution', fontweight='bold')
plt.ylabel('Count')
plt.show()


In [ ]:
# Train/test split (stratified to preserve class proportions)
X = df.drop(columns=['Outcome'])
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f'Train: {X_train.shape[0]} rows ({y_train.mean()*100:.1f}% positive)')
print(f'Test : {X_test.shape[0]} rows ({y_test.mean()*100:.1f}% positive)')


**Observation.** The dataset is moderately imbalanced (~35% positive / ~65%
negative). We use **stratified splitting** so that both train and test sets keep
the same class ratio — otherwise the test set could have very few positives and
the metrics would become noisy.


## Exercise 2: Model picking and standardization

### Which classification model and why?
We pick **Logistic Regression** because:

- The target is **binary** (`Outcome ∈ {0, 1}`) — exactly what logistic regression
  models out of the box.
- It is **interpretable**: each coefficient translates to a log-odds change per
  unit of the feature, which is important in clinical contexts (clinicians want to
  know *why* the model flagged a patient).
- It is a **strong baseline** for tabular binary classification — a complex model
  is only worth its cost if it clearly beats this baseline.
- It produces **calibrated probabilities** directly, useful for setting decision
  thresholds based on the cost of false negatives vs false positives.

### Do we need to standardize?
**Yes.** Logistic regression with L2 regularization (the sklearn default) is
**sensitive to feature scale**: features with larger numeric ranges (like `Insulin`
and `Glucose`, which span 0–800 and 0–200) would dominate the loss compared with
smaller-range features (like `BMI` ~ 0–60 or `DiabetesPedigreeFunction` ~ 0–2.5).
We use `StandardScaler` to give every feature mean 0 and std 1, and we **fit it
only on the training set** to avoid leakage.


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)
print('After scaling — mean and std of training features:')
print(scaled_df.agg(['mean', 'std']).round(3))


## Exercise 3: Train the Logistic Regression model


In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

# Show the learned coefficients (interpretable: log-odds per std of feature)
coef_df = pd.Series(model.coef_[0], index=X.columns).sort_values(key=abs, ascending=False)
print('Logistic regression coefficients (sorted by |value|):')
print(coef_df.round(3))

print(f'\nIntercept: {model.intercept_[0]:.3f}')


**Interpretation.** The largest positive coefficients (Glucose, BMI, Age,
Pregnancies, DiabetesPedigreeFunction) all increase the predicted probability of
diabetes — clinically reasonable. `BloodPressure` and `Insulin` have smaller
effects after controlling for the others.


## Exercise 4: Evaluation metrics

We compute accuracy, confusion matrix, precision, recall, F1.


In [ ]:
y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f'Accuracy : {acc:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall   : {rec:.4f}')
print(f'F1-score : {f1:.4f}')


In [ ]:
# Accuracy bar (single value, contextualized against the majority-class baseline)
baseline = y_test.value_counts(normalize=True).max()

plt.figure(figsize=(7, 3.5))
plt.barh(['Majority-class baseline', 'Logistic Regression'],
         [baseline, acc],
         color=['lightgray', 'steelblue'])
plt.xlim(0, 1)
plt.xlabel('Accuracy')
plt.title('Accuracy vs majority-class baseline', fontweight='bold')
for i, v in enumerate([baseline, acc]):
    plt.text(v + 0.01, i, f'{v:.3f}', va='center', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Lift over baseline: +{(acc - baseline) * 100:.1f} percentage points')


**Accuracy comment.** Accuracy alone is misleading on imbalanced data: a model
that always predicts 'negative' would already score ~65%. Our model improves on
that meaningfully, but the gap is what matters, not the absolute number.


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(cm, display_labels=['Negative', 'Positive'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion matrix — test set', fontweight='bold')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negatives (TN): {tn}  | False Positives (FP): {fp}')
print(f'False Negatives (FN): {fn} | True Positives (TP):  {tp}')


**Confusion matrix comment.** In a medical context, the **False Negatives** are
the most costly errors: a diabetic patient sent home untreated. If FN seems too
high, we should lower the decision threshold (currently 0.5) to trade some
precision for more recall.


In [ ]:
# Precision / Recall / F1 as a grouped bar (per class) + overall classification report
report = classification_report(y_test, y_pred, target_names=['Negative', 'Positive'], output_dict=True)
report_df = pd.DataFrame(report).transpose().round(3)
print(report_df)

metrics_df = pd.DataFrame({
    'Negative': [report['Negative']['precision'], report['Negative']['recall'], report['Negative']['f1-score']],
    'Positive': [report['Positive']['precision'], report['Positive']['recall'], report['Positive']['f1-score']],
}, index=['Precision', 'Recall', 'F1-score'])

ax = metrics_df.plot(kind='bar', figsize=(8, 4.5), color=['steelblue', 'tomato'], edgecolor='white')
plt.title('Precision / Recall / F1 per class', fontweight='bold')
plt.ylim(0, 1)
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.legend(title='Class')
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3)
plt.tight_layout()
plt.show()


**Precision/Recall/F1 comment.** Precision tends to be higher than recall on the
**positive** class, which means: when the model says 'diabetic' it is usually
right, but it misses some true diabetics. F1 balances these two and is the
metric we would primarily report. To improve recall on the positive class we
could lower the threshold below 0.5 or use `class_weight='balanced'`.


## Exercise 5: Decision boundary visualization

Our model uses 8 features, so the decision boundary lives in 8-dimensional space.
To visualize it we **project to 2D using PCA** (the first two principal
components), train a logistic regression on those 2 components only, and plot
the boundary on the test set. The reported accuracy is the accuracy of the
**2D model** (necessarily lower than the full 8-feature model).


In [ ]:
# Fit PCA on the scaled training data and project both sets
pca = PCA(n_components=2, random_state=42)
X_train_2d = pca.fit_transform(X_train_scaled)
X_test_2d = pca.transform(X_test_scaled)

# Train a 2D logistic regression for the boundary
model_2d = LogisticRegression(max_iter=1000, random_state=42)
model_2d.fit(X_train_2d, y_train)
acc_2d = accuracy_score(y_test, model_2d.predict(X_test_2d))

# Build a meshgrid covering the data
x_min, x_max = X_test_2d[:, 0].min() - 1, X_test_2d[:, 0].max() + 1
y_min, y_max = X_test_2d[:, 1].min() - 1, X_test_2d[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))
Z = model_2d.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)

plt.figure(figsize=(9, 6))
plt.contourf(xx, yy, Z, levels=20, cmap='RdBu_r', alpha=0.6)
plt.colorbar(label='P(diabetes)')
plt.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=1.5)
scatter = plt.scatter(X_test_2d[:, 0], X_test_2d[:, 1],
                      c=y_test, cmap='RdBu_r', edgecolor='white', s=50)
plt.xlabel('PC 1')
plt.ylabel('PC 2')
plt.title(f'Decision boundary (PCA 2D) — 2D model accuracy: {acc_2d:.3f}',
          fontweight='bold')
plt.legend(*scatter.legend_elements(), title='Outcome', loc='upper right')
plt.tight_layout()
plt.show()

print(f'Full 8-feature model accuracy: {acc:.3f}')
print(f'PCA 2D model accuracy:         {acc_2d:.3f}')


## Exercise 6: ROC curve

The ROC curve plots True Positive Rate (Recall) against False Positive Rate at
every possible threshold. The AUC summarizes it in one number — 0.5 = random,
1.0 = perfect. ROC is **threshold-independent** and robust to class imbalance.


In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='steelblue', linewidth=2.5, label=f'Logistic Regression (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random classifier')
plt.fill_between(fpr, tpr, alpha=0.15, color='steelblue')
plt.xlim(0, 1)
plt.ylim(0, 1.02)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve — Diabetes prediction', fontweight='bold')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'ROC-AUC: {auc:.4f}')


**ROC comment.** An AUC around 0.83 (typical for this dataset and model) means
that, for a random pair of one diabetic and one non-diabetic patient, the model
ranks the diabetic patient as more risky **~83% of the time**. That is a solid
baseline. To push it further we could try Gradient Boosting, engineer interaction
features (e.g., BMI × Age), or tune the threshold to match the clinical cost of
false negatives.


## Summary

- Binary classification with a clear positive class (diabetes), moderate imbalance.
- **Logistic Regression with `StandardScaler`** is the right baseline: fast,
  interpretable, calibrated, and meets clinical explainability requirements.
- Reported metrics on the held-out test set: accuracy, precision, recall, F1,
  confusion matrix, and ROC-AUC.
- The decision boundary plot in PCA space gives a visual intuition of how the
  model separates the two classes.
- Next steps for production: tune threshold against business cost of FN vs FP,
  benchmark against XGBoost/LightGBM, add calibration plots, and monitor
  performance drift after deployment.
